In [5]:
import polars as pl
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
import time
import os
import math
from afinn import Afinn
from collections import defaultdict
import polars.selectors as cs
import re
import concurrent.futures

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/javclamar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
def cargar_nrc_dict(path="../data/lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"):
    """
    Carga el diccionario NRC en memoria.
    """
    nrc_map = defaultdict(set)
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 3 and int(parts[2]) == 1:
                    nrc_map[parts[0]].add(parts[1])
    except FileNotFoundError:
        print(f"ADVERTENCIA: No se encontró el archivo NRC en {path}. NRC devolverá 0.")
    except Exception as e:
        print(f"Error cargando NRC: {e}")
    return nrc_map

def calcular_vader(texts):
    """
    Calcula los scores de VADER para un conjunto de textos y devuelve el compound.
    """
    sia = SentimentIntensityAnalyzer()
    return [sia.polarity_scores(str(t))['compound'] for t in texts]

def calcular_afinn(texts):
    """
    Calcula score AFINN normalizado tal como se hace en VADER para un conjunto de textos.
    """
    afinn = Afinn()
    scores = []
    alpha = 15
    
    for t in texts:
        raw = afinn.score(str(t))
        
        if raw != 0:
            norm = raw / math.sqrt((raw**2) + alpha)
        else:
            norm = 0.0
        scores.append(norm)
    return scores

def calcular_nrc(texts):
    """
    Calcula densidad de emociones NRC para un conjunto de textos.
    """
    emotions = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 
                'sadness', 'surprise', 'trust', 'positive', 'negative']
    
    results = {f'nrc_{e}': [] for e in emotions}
    
    tokenizer = re.compile(r'\b[a-z]+\b')

    if not NRC_DICT:
        empty_list = [0.0] * len(texts)
        for k in results: results[k] = empty_list
        return results

    for text in texts:
        text_str = str(text).lower()
        
        words = tokenizer.findall(text_str)
        total_words = len(words)
        
        counts = {e: 0.0 for e in emotions}
        
        if total_words > 0:
            for w in words:
                if w in NRC_DICT:
                    found_emotions = NRC_DICT[w]
                    for e in found_emotions:
                        counts[e] += 1
            
            for e in emotions:
                counts[e] = counts[e] / total_words
        
        for e in emotions:
            results[f'nrc_{e}'].append(counts[e])
            
    return results

LEXICONS_AVAILABLE = {
    "vader_score": calcular_vader,
    "afinn_score": calcular_afinn,
    "nrc_emotions": calcular_nrc
}

NRC_DICT = cargar_nrc_dict() 

def process_chunk(args):
    """
    Función que ejecutan los nucleos.
    """
    texts, active_models = args
    results = {}
    
    for model_name in active_models:
        if model_name in LEXICONS_AVAILABLE:
            try:
                results[model_name] = LEXICONS_AVAILABLE[model_name](texts)
            except Exception as e:
                print(f"Error en {model_name}: {e}")
                results[model_name] = [0.0] * len(texts)
                
    return results

In [ ]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
csv_reviews_output_scores = '../results/yelp_academic_dataset_review_scored.csv'
batch_size = 200000
total_rows = 6_990_280


def analyze_sentiment(input_csv, output_csv, lexicons=['vader_score', 'afinn_score', 'nrc_emotions']):
    
    if os.path.exists(output_csv):
        try:
            df_check = pl.read_csv(output_csv, n_rows=1, ignore_errors=True)
            if df_check.height == 0:
                source_csv = input_csv
                temp_output_csv = output_csv
            else:
                print(f"Leyendo datos existentes de: {output_csv}")
                source_csv = output_csv
                temp_output_csv = output_csv + ".tmp"
        except Exception:
            source_csv = input_csv
            temp_output_csv = output_csv
    else:
        source_csv = input_csv
        temp_output_csv = output_csv

    dummy_output = process_chunk((["test"], lexicons))
    
    new_lexicon_cols = []
    for lexicon in lexicons:
        res = dummy_output[lexicon]
        if isinstance(res, dict):
            new_lexicon_cols.extend(sorted(res.keys()))
        else:
            new_lexicon_cols.append(lexicon)
            
    try:
        df_schema = pl.read_csv(source_csv, n_rows=1, ignore_errors=True)
        existing_columns = df_schema.columns
        
        final_columns = list(dict.fromkeys(existing_columns + new_lexicon_cols))
        
        with open(temp_output_csv, 'w') as f:
            f.write(",".join([f'"{c}"' for c in final_columns]) + "\n")
            
    except Exception as e:
        print(f"Error gestionando headers: {e}")
        return

    num_cores = os.cpu_count()
    
    reader = pl.read_csv_batched(source_csv, batch_size=batch_size, ignore_errors=True)
    start_time = time.time()
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
        with tqdm(total=total_rows, unit="reviews", desc="Procesando") as pbar:
            while True:
                batches = reader.next_batches(1)
                if not batches: break
                
                df_batch = batches[0]
                
                if "text" not in df_batch.columns:
                    raise ValueError("El archivo fuente no tiene la columna 'text' necesaria para recalcular.")

                texts = df_batch["text"].to_list()
                
                chunk_size = math.ceil(len(texts) / num_cores)
                chunks = [texts[i:i + chunk_size] for i in range(0, len(texts), chunk_size)]
                worker_args = [(chunk, lexicons) for chunk in chunks]
                
                results_generator = executor.map(process_chunk, worker_args)
                
                batch_data_flat = {col: [] for col in new_lexicon_cols}
                
                for res_dict in results_generator:
                    for lexicon in lexicons:
                        output = res_dict[lexicon]
                        
                        if isinstance(output, dict):
                            for sub_col in output:
                                batch_data_flat[sub_col].extend(output[sub_col])
                        else:
                            batch_data_flat[lexicon].extend(output)
                
                df_scored = df_batch.with_columns(
                    [pl.Series(name=col, values=batch_data_flat[col], dtype=pl.Float64) for col in new_lexicon_cols]
                )
                
                df_scored.select(final_columns).write_csv(
                    file=open(temp_output_csv, "a"),
                    include_header=False,
                    quote_style="always"
                )
                
                pbar.update(len(texts))

    print(f"Procesamiento finalizado en: {(time.time() - start_time) / 60:.2f} min")

    if os.path.exists(output_csv) and source_csv == output_csv:
        os.remove(output_csv)
        os.rename(temp_output_csv, output_csv)

analyze_sentiment(csv_reviews, csv_reviews_output_scores, lexicons=['vader_score', 'afinn_score', 'nrc_emotions'])

Output ../results/yelp_academic_dataset_review_scored.csv exists but is empty — reading raw input ../data/csv/yelp_academic_dataset_review.csv


Procesando: 100%|██████████| 6990280/6990280 [40:39<00:00, 2865.15reviews/s]

Procesamiento finalizado en: 40.66 min


In [8]:
csv_reviews_output_scores = '../results/yelp_academic_dataset_review_scored.csv'

print(pl.scan_csv(csv_reviews_output_scores, ignore_errors=True)
    .select(cs.numeric())
    .tail(5).collect())

shape: (5, 14)
┌───────┬───────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ stars ┆ num_chars ┆ vader_scor ┆ afinn_scor ┆ … ┆ nrc_positi ┆ nrc_sadne ┆ nrc_surpr ┆ nrc_trust │
│ ---   ┆ ---       ┆ e          ┆ e          ┆   ┆ ve         ┆ ss        ┆ ise       ┆ ---       │
│ i64   ┆ i64       ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ f64       │
│       ┆           ┆ f64        ┆ f64        ┆   ┆ f64        ┆ f64       ┆ f64       ┆           │
╞═══════╪═══════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 5     ┆ 322       ┆ 0.1027     ┆ -0.612372  ┆ … ┆ 0.068966   ┆ 0.0       ┆ 0.0       ┆ 0.086207  │
│ 5     ┆ 397       ┆ 0.8549     ┆ 0.840168   ┆ … ┆ 0.050633   ┆ 0.0       ┆ 0.012658  ┆ 0.012658  │
│ 4     ┆ 467       ┆ 0.6792     ┆ 0.840168   ┆ … ┆ 0.034483   ┆ 0.0       ┆ 0.011494  ┆ 0.011494  │
│ 5     ┆ 2317      ┆ 0.9982     ┆ 0.998055   ┆ … ┆ 0.079487   ┆ 0.010256  ┆